# 14 years of demographic change for any US ZIP code — in 10 lines of Python

Every US ZIP code has a longitudinal demographic record: the US Census Bureau's
**American Community Survey (ACS) 5-Year Estimates**, published annually since 2011.
Getting at it normally means downloading Census tables year by year, reconciling
variable codes that move between years, and handling the 2020 `GEO_ID` format change.

This notebook pulls a clean 14-year series (2011–2024) for one ZIP code with a
**single API call** to the [ZIP Codes API](https://api.zip-codes.com/docs) — the API
normalizes the year-to-year variable drift for you.

**No signup needed to run this notebook**: it uses the public demo key, which works for
demo ZIPs (`90210`, `10001`, `10118`, `32504`, `00601`, `96950`, `33139`, `60601`,
`98101`, `30301`). For **any** US ZIP, [grab a free key](https://www.zip-codes.com/api/signup)
— 2,500 credits/day, no credit card.

In [1]:
import requests, pandas as pd

ZIP, KEY = "90210", "zc_test_DEMOAPIKEY000000000000"  # demo key: demo ZIPs only
flags = ",".join(f"acs_demographic_{y}" for y in range(2011, 2025))
resp = requests.get("https://api.zip-codes.com/v2/zip",
                    params={"code": ZIP, "include": flags},
                    headers={"X-Api-Key": KEY}, timeout=60).json()
sa = {v["year"]: v["demographic"]["sex_and_age"]
      for v in resp["results"][0]["acs"].values()}
df = pd.DataFrame({"population": {y: s["total_population"]["est"] for y, s in sa.items()},
                   "median_age": {y: s["median_age"]["est"] for y, s in sa.items()}}).sort_index()
print(df)

      population  median_age
2011       21719        45.7
2012       21517        47.0
2013       21548        46.4
2014       21478        46.8
2015       22052        46.1
2016       20957        47.5
2017       20016        47.1
2018       19909        48.1
2019       19314        49.2
2020       19229        49.0
2021       19627        49.1
2022       19180        50.9
2023       19652        51.2
2024       19004        51.9


Beverly Hills is aging: median age up **6.2 years** since 2011 while population fell ~12%.
One call, one chart:

In [2]:
ax = df.plot(secondary_y="median_age", figsize=(9, 4),
             title=f"ZIP {ZIP}: 14 years of demographic change")
ax.set_ylabel("population"); ax.right_ax.set_ylabel("median age");

## Going further

The same call pattern works for **all four ACS profiles**, each with the full 2011–2024
back-catalog (~500 fields per ZIP per year in recent vintages):

| Flag | Profile | Highlights |
|---|---|---|
| `acs_demographic_YYYY` | DP05 | population, age structure, race, ethnicity |
| `acs_social_YYYY` | DP02 | households, education, language, ancestry, internet access |
| `acs_economic_YYYY` | DP03 | income, employment, poverty, occupation, commute |
| `acs_housing_YYYY` | DP04 | home values, rent, occupancy, tenure, structure age |

Every value comes with its **margin of error** (`moe`), and percent fields carry
`pct` / `pct_moe` — so you can do the analysis correctly. Census-suppressed values
are explicit `null`s, never zeros.

Useful asks:
- *Where did rent rise fastest?* → `acs_housing_2011` vs `acs_housing_2024`, `rent.median`
- *Which ZIPs aged most?* → `acs_demographic_YYYY`, `sex_and_age.median_age`
- *Education shifts* → `acs_social_YYYY`, `education.bachelors_or_higher`

Docs: <https://api.zip-codes.com/docs> · Machine-readable: <https://api.zip-codes.com/llms-full.txt>

## Bundled sample: every Nevada ZIP × 14 years

This repo ships [`data/nevada_acs_by_zip_2011_2024.csv`](data/nevada_acs_by_zip_2011_2024.csv)
— 182 Nevada ZCTAs × 2011–2024, ten headline indicators with margins of error
(see the [README](README.md) for the data dictionary). Nevada's post-recession arc is
dramatic: downtown Las Vegas (89101) median household income rose 53% while median
home values more than doubled.

In [3]:
nv = pd.read_csv("data/nevada_acs_by_zip_2011_2024.csv")
vegas = nv[nv.zcta == 89101].set_index("year")
ax = vegas[["median_household_income_est", "median_home_value_est"]].plot(
    secondary_y="median_home_value_est", figsize=(9, 4),
    title="ZCTA 89101 (downtown Las Vegas): income vs home value, 2011-2024")
ax.set_ylabel("median household income ($)"); ax.right_ax.set_ylabel("median home value ($)");

## Notes, credits, citation

- **ZIP vs ZCTA**: ZIP-level demographics use Census ZCTA approximations — the standard
  approach for non-PO-Box ZIPs, with known caveats for very small or PO-Box-only ZIPs.
- **Source data**: US Census Bureau, American Community Survey 5-Year Estimates
  (public domain). The API normalizes year-to-year variable drift and GEO_ID format changes.
- **Free access**: 2,500 credits/day free tier ([signup](https://www.zip-codes.com/api/signup)).
  Researchers, journalists, and students can request extended credit grants in exchange
  for a data acknowledgement — contact <jharris@zip-codes.com>.

Suggested acknowledgement:

> Demographic data: US Census Bureau ACS 5-Year Estimates, accessed via ZIP Codes API
> (Zip-Codes.com), https://www.zip-codes.com/api/